In [17]:
print("hello")

hello


In [1]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [2]:
import micrograd.engine as engine 
import micrograd.nn as nn

In [3]:
n = nn.MLP(3, [4, 4, 1])

In [10]:
n.layers[0].neurons[0].w[0]

Value(data=-0.37046445103555525, grad=0)

In [11]:
n.layers[1]

Layer of [ReLUNeuron(4), ReLUNeuron(4), ReLUNeuron(4), ReLUNeuron(4)]

In [12]:
n.layers[1].neurons[1].w

[Value(data=-0.7880088891663983, grad=0),
 Value(data=0.46741164252556344, grad=0),
 Value(data=-0.8516102217398751, grad=0),
 Value(data=0.8574864450239532, grad=0)]

In [14]:
# forward pass 
# for x in xs: 
#     print(n(x))
ypreds = [n(x) for x in xs]

In [16]:
import pprint
pprint.pprint(ypreds)

[Value(data=-0.24689561970947324, grad=0),
 Value(data=-0.32553516246494696, grad=0),
 Value(data=-0.9250787139980299, grad=0),
 Value(data=0.024498906392112302, grad=0)]


In [33]:
# loss = 0.0 
# for ypred, y in zip(ypreds, ys): 
#     # print(ypred, y)
#     # print((y-ypred) ** 2)
#     loss += (y - ypred) ** 2 

loss += sum((y - ypred) ** 2 for ypred, y in zip(ypreds, ys))

In [34]:
print(loss)

Value(data=2.966867086248231, grad=0)


In [35]:
len(n.parameters())

41

In [48]:
# print(n.layers[0])

In [47]:
# len(n.layers[2].parameters())

In [49]:
loss.backward()

In [54]:
print(n.layers[0].neurons[1].w[0])

Value(data=0.02991298618403926, grad=1.4542287573960468)


In [55]:
# loss +ve 
# weight +ve
# slope +ve

In [56]:
for p in n.parameters(): 
    p.data += (-0.01 * p.grad)

In [ ]:
ypreds = [n(x) for x in xs]
loss = sum((y - ypred) ** 2 for ypred, y in zip(ypreds, ys))
for p in n.parameters(): 
    p.grad = 0.0
loss.backward() 
for p in n.parameters(): 
    p.data += (-0.01 * p.grad)

In [ ]:
ypreds = [n(x) for x in xs]
loss = sum((y - ypred) ** 2 for ypred, y in zip(ypreds, ys))
print(loss)

Value(data=0.18832417500886187, grad=0)


In [4]:
# 1000 epochs
for i in range(1000): 

    # forward pass 
    ypreds = [n(x) for x in xs]

    # loss
    loss = sum((y - ypred) ** 2 for ypred, y in zip(ypreds, ys))

    # logging 
    print(i, loss)

    # zero grad 
    # for p in n.parameters(): 
    #     p.grad = 0.0
    n.zero_grad()

    # backward pass 
    loss.backward() 

    # update weights 
    for p in n.parameters(): 
        p.data += (-0.01 * p.grad)

0 Value(data=9.586700305177738, grad=0)
1 Value(data=4.753599648396821, grad=0)
2 Value(data=4.302110010011654, grad=0)
3 Value(data=4.161001640158783, grad=0)
4 Value(data=4.0858441719920995, grad=0)
5 Value(data=4.032389570728014, grad=0)
6 Value(data=3.9891936472141225, grad=0)
7 Value(data=3.949305484083857, grad=0)
8 Value(data=3.908603081544785, grad=0)
9 Value(data=3.863889853454388, grad=0)
10 Value(data=3.8121435170861977, grad=0)
11 Value(data=3.75015702256641, grad=0)
12 Value(data=3.6743345860737295, grad=0)
13 Value(data=3.580578417220476, grad=0)
14 Value(data=3.465395290008673, grad=0)
15 Value(data=3.3418340117528333, grad=0)
16 Value(data=3.19663791521449, grad=0)
17 Value(data=3.0273119674088007, grad=0)
18 Value(data=2.8324304807081724, grad=0)
19 Value(data=2.6123495081429757, grad=0)
20 Value(data=2.3700151988208282, grad=0)
21 Value(data=2.111641400752913, grad=0)
22 Value(data=1.846875734079501, grad=0)
23 Value(data=1.5880102131393912, grad=0)
24 Value(data=1.34

In [5]:
ypreds = [n(x) for x in xs]

In [6]:
for y, ypred in zip(ys, ypreds): 
    print(y, ypred)

1.0 Value(data=1.0, grad=0)
-1.0 Value(data=-1.0, grad=0)
-1.0 Value(data=-0.9999999999999999, grad=0)
1.0 Value(data=0.9999999999999998, grad=0)


In [7]:
import pickle
import dill
dill.dump(n, open('lect24_model_micrograd.pkl', 'wb'))

# loaded_model = dill.load(open('lect24_model_micrograd.pkl', 'rb'))

In [84]:
loaded_model = dill.load(open('lect24_model_micrograd.pkl', 'rb'))

In [85]:
ypreds = [loaded_model(x) for x in xs]
for y, ypred in zip(ys, ypreds): 
    print(y, ypred)

1.0 Value(data=1.0000000000000004, grad=0)
-1.0 Value(data=-1.0, grad=0)
-1.0 Value(data=-0.9999999999999998, grad=0)
1.0 Value(data=0.9999999999999974, grad=0)


In [86]:
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir}) #, node_attr={'rankdir': 'TB'})
    
    for n in nodes:
        dot.node(name=str(id(n)), label = "{data %.4f | grad %.4f  }" % (n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dot

In [88]:
# draw_dot(n([2.0, 3.0, -1.0]))

In [9]:
import torch
import torch.nn as nn

In [10]:
xs = torch.tensor([
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
])
ys = torch.tensor([[1.0], [-1.0], [-1.0], [1.0]]) # desired targets

In [11]:
xs.shape

torch.Size([4, 3])

In [12]:
ys.shape

torch.Size([4, 1])

In [13]:
n = nn.Sequential(
    nn.Linear(3, 4), 
    nn.ReLU(), 
    nn.Linear(4, 4), 
    nn.ReLU(), 
    nn.Linear(4, 1)
)

In [23]:
ypreds = n(xs)

In [24]:
ypreds

tensor([[ 0.2543],
        [ 0.1516],
        [-0.1682],
        [-0.0610]], grad_fn=<AddmmBackward0>)

In [25]:
ypreds.shape

torch.Size([4, 1])

In [26]:
ys.shape

torch.Size([4, 1])

In [27]:
ys

tensor([[ 1.],
        [-1.],
        [-1.],
        [ 1.]])

In [28]:
loss = ((ys - ypreds) ** 2).sum()

In [29]:
loss

tensor(3.6999, grad_fn=<SumBackward0>)

In [30]:
n.zero_grad()

In [31]:
loss.backward()

In [ ]:
with torch.no_grad(): 
    for p in n.parameters(): 
        p.data += (-0.01 * p.grad)

In [33]:
ypreds = n(xs)

In [34]:
loss = ((ys - ypreds) ** 2).sum()

In [36]:
loss

tensor(3.3099, grad_fn=<SumBackward0>)

In [54]:
# (n[0].weight[0].grad)

In [52]:
n[0].weight.grad

tensor([[-0.4263, -0.6395,  0.2132],
        [ 1.0284, -5.0705,  2.7148],
        [-0.6019,  1.6517, -1.4675],
        [-0.0967, -0.1934, -0.1934]])

In [53]:
n[0].weight

Parameter containing:
tensor([[-0.1543,  0.1383, -0.5665],
        [ 0.4913,  0.0288, -0.3876],
        [ 0.1405, -0.1774, -0.1577],
        [-0.3170, -0.2393,  0.4685]], requires_grad=True)

In [14]:
for i in range(1000): 

    # forward pass
    ypreds = n(xs)

    # loss
    loss = ((ys - ypreds) ** 2).sum()

    # logging
    print(i, loss)

    # zero grad 
    n.zero_grad() 

    # backward pass 
    loss.backward() 

    # update weights
    with torch.no_grad(): 
        for p in n.parameters(): 
            p.data += (-0.01 * p.grad)

0 tensor(4.3055, grad_fn=<SumBackward0>)
1 tensor(4.1166, grad_fn=<SumBackward0>)
2 tensor(3.9537, grad_fn=<SumBackward0>)
3 tensor(3.8430, grad_fn=<SumBackward0>)
4 tensor(3.7440, grad_fn=<SumBackward0>)
5 tensor(3.6503, grad_fn=<SumBackward0>)
6 tensor(3.5579, grad_fn=<SumBackward0>)
7 tensor(3.4738, grad_fn=<SumBackward0>)
8 tensor(3.3857, grad_fn=<SumBackward0>)
9 tensor(3.2987, grad_fn=<SumBackward0>)
10 tensor(3.2034, grad_fn=<SumBackward0>)
11 tensor(3.1005, grad_fn=<SumBackward0>)
12 tensor(3.0006, grad_fn=<SumBackward0>)
13 tensor(2.8996, grad_fn=<SumBackward0>)
14 tensor(2.7972, grad_fn=<SumBackward0>)
15 tensor(2.6927, grad_fn=<SumBackward0>)
16 tensor(2.5863, grad_fn=<SumBackward0>)
17 tensor(2.4813, grad_fn=<SumBackward0>)
18 tensor(2.3786, grad_fn=<SumBackward0>)
19 tensor(2.2788, grad_fn=<SumBackward0>)
20 tensor(2.1657, grad_fn=<SumBackward0>)
21 tensor(2.0614, grad_fn=<SumBackward0>)
22 tensor(1.9746, grad_fn=<SumBackward0>)
23 tensor(1.8705, grad_fn=<SumBackward0>)
24

In [15]:
ypreds = n(xs)
ypreds

tensor([[ 1.0000],
        [-1.0000],
        [-1.0000],
        [ 1.0000]], grad_fn=<AddmmBackward0>)

In [16]:
ys

tensor([[ 1.],
        [-1.],
        [-1.],
        [ 1.]])

In [17]:
sum(p.numel() for p in n.parameters())

41

In [65]:
# type(n.parameters())

In [67]:
# n.parameters().numel()

In [ ]:
import pickle
import dill
dill.dump(n, open('lect24_model_pytorch.pkl', 'wb'))

# loaded_model = dill.load(open('lect24_model_micrograd.pkl', 'rb'))

In [19]:
loaded_model = dill.load(open('lect24_model_pytorch.pkl', 'rb'))

In [20]:
ypreds = loaded_model(xs)

In [21]:
ypreds

tensor([[ 1.0000],
        [-1.0000],
        [-1.0000],
        [ 1.0000]], grad_fn=<AddmmBackward0>)